# 03 — Ground Truth Generation

For all movies in `movies_clean.csv`, generate 3 natural user questions using each model.
Saves progress every `CHUNK_SIZE` movies — safe to interrupt and resume.

Outputs one file per model:
- `data/ground-truth-retrieval-gpt-5.4-mini.csv`
- `data/ground-truth-retrieval-gpt-5.6-luna.csv`

Each file has columns `movie_id`, `question`.

In [1]:
import sys, os, time, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv(ROOT / '.envrc')

from openai import OpenAI
from google import genai as google_genai

df = pd.read_csv(ROOT / 'data/movies_clean.csv')
print(f'Loaded {len(df)} movies')

Loaded 2000 movies


In [ ]:
CHUNK_SIZE = 100
MAX_RETRIES = 6   # waits: 15, 30, 60, 120, 240, 480s
BASE_WAIT = 15

PROMPT_TEMPLATE = """Given this movie:
Title: {title}
Genres: {genres}
Overview: {overview}

Generate exactly 3 natural user questions that someone might type into a movie recommendation
assistant that would make this movie a relevant answer.
Output as a JSON array of 3 strings and nothing else."""


def generate_openai(row, client, model):
    prompt = PROMPT_TEMPLATE.format(
        title=row['title'], genres=row['genres'], overview=str(row['overview'])[:400]
    )
    resp = client.responses.create(
        model=model,
        input=[{'role': 'user', 'content': prompt}],
    )
    return json.loads(resp.output_text)

def generate_for_model(df, model_name, generate_fn, chunk_size=CHUNK_SIZE, delay=0.1):
    out_path = ROOT / f'data/ground-truth-retrieval-{model_name}.csv'

    if out_path.exists():
        done = set(pd.read_csv(out_path)['movie_id'].astype(str))
        print(f'Resuming — {len(done)} movie IDs already done')
    else:
        done = set()

    remaining = df[~df['id'].astype(str).isin(done)].reset_index(drop=True)
    print(f'{len(remaining)} movies left to process')

    records = []
    for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc=model_name)):
        try:
            questions = generate_fn(row)
            for q in questions:
                records.append({'movie_id': row['id'], 'question': q})
            time.sleep(delay)
        except Exception as e:
            print(f'  skipping {row["title"]}: {e}')

        if records and (len(records) >= chunk_size * 3 or i == len(remaining) - 1):
            pd.DataFrame(records).to_csv(
                out_path, mode='a', header=not out_path.exists(), index=False
            )
            records = []
            print(f'  chunk saved ({i+1}/{len(remaining)})')

    gt = pd.read_csv(out_path)
    print(f'Total: {len(gt)} pairs for {model_name}')
    return gt

In [9]:
# --- gpt-5.4-mini ---
openai_client = OpenAI()
gt_mini = generate_for_model(
    df, 'gpt-5.4-mini',
    lambda row: generate_openai(row, openai_client, 'gpt-5.4-mini')
)
gt_mini.head(3)

2000 movies left to process


gpt-5.4-mini:   0%|          | 0/2000 [00:00<?, ?it/s]

  chunk saved (100/2000)
  chunk saved (200/2000)
  chunk saved (300/2000)
  chunk saved (400/2000)
  chunk saved (500/2000)
  chunk saved (600/2000)
  chunk saved (700/2000)
  chunk saved (800/2000)
  chunk saved (900/2000)
  chunk saved (1000/2000)
  chunk saved (1100/2000)
  chunk saved (1200/2000)
  chunk saved (1300/2000)
  chunk saved (1400/2000)
  chunk saved (1500/2000)
  chunk saved (1600/2000)
  chunk saved (1700/2000)
  chunk saved (1800/2000)
  chunk saved (1900/2000)
  chunk saved (2000/2000)
Total: 6000 pairs for gpt-5.4-mini


,movie_id,question
0,157336,Can you recommend a science fiction adventure ...
1,157336,What’s a good drama movie with an epic journey...
2,157336,Suggest a movie about explorers trying to save...


In [11]:
# --- gpt-5.6-luna ---
gt_luna = generate_for_model(
    df, 'gpt-5.6-luna',
    lambda row: generate_openai(row, openai_client, 'gpt-5.6-luna')
)
gt_luna.head(3)

2000 movies left to process


gpt-5.6-luna:   0%|          | 0/2000 [00:00<?, ?it/s]

  chunk saved (100/2000)
  chunk saved (200/2000)
  chunk saved (300/2000)
  chunk saved (400/2000)
  chunk saved (500/2000)
  chunk saved (600/2000)
  chunk saved (700/2000)
  chunk saved (800/2000)
  chunk saved (900/2000)
  chunk saved (1000/2000)
  chunk saved (1100/2000)
  chunk saved (1200/2000)
  chunk saved (1300/2000)
  chunk saved (1400/2000)
  chunk saved (1500/2000)
  chunk saved (1600/2000)
  chunk saved (1700/2000)
  chunk saved (1800/2000)
  chunk saved (1900/2000)
  chunk saved (2000/2000)
Total: 6000 pairs for gpt-5.6-luna


,movie_id,question
0,157336,Can you recommend a thought-provoking science ...
1,157336,What are some epic space adventure films that ...
2,157336,I’m looking for an emotionally powerful sci-fi...


In [6]:
# Sanity check — OpenAI ground truth files
for model in ['gpt-5.4-mini', 'gpt-5.6-luna']:
    path = ROOT / f'data/ground-truth-retrieval-{model}.csv'
    if path.exists():
        gt = pd.read_csv(path)
        print(f'{model}: {len(gt)} pairs, {gt["movie_id"].nunique()} unique movies')
    else:
        print(f'{model}: not yet generated')

gpt-5.4-mini: 6000 pairs, 2000 unique movies
gpt-5.6-luna: 6000 pairs, 2000 unique movies
